In [1]:
import os
import pickle

import numpy as np
import pandas as pd

from pycisTopic.pseudobulk_peak_calling import export_pseudobulk, peak_calling

/home/jovyan/pycisTopic/src/pycisTopic/__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
/opt/python4Jup/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-08-20 21:01:37,360	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
NOTEBOOK_DIR = os.getcwd()
REPO_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, ".."))

SAMPLE = "8k_mouse_cortex_ATACv2_nextgem_Chromium_Controller"
DEPENDENCIES = "/home/jovyan/work/PUMATAC_dependencies"

CISTOPIC_OUT = os.path.join(REPO_ROOT, "results", "cistopic")
CTO_PATH = os.path.join(CISTOPIC_OUT, f"{SAMPLE}__cto_with_model.pkl")

FRAGMENTS_PATH = os.path.join(
    NOTEBOOK_DIR, "notebooks_PUMATAC", "out", "data", "fragments",
    f"{SAMPLE}.fragments.tsv.gz",
)
CHROMSIZES = os.path.join(DEPENDENCIES, "genomes", "mm10.chrom.sizes")

PSEUDOBULK_OUT = os.path.join(REPO_ROOT, "results", "pseudobulk")
BED_PATH = os.path.join(PSEUDOBULK_OUT, "bed")
BIGWIG_PATH = os.path.join(PSEUDOBULK_OUT, "bigwig")
MACS_OUT = os.path.join(REPO_ROOT, "results", "macs2")
for d in [PSEUDOBULK_OUT, BED_PATH, BIGWIG_PATH, MACS_OUT]:
    os.makedirs(d, exist_ok=True)

with open(CTO_PATH, "rb") as f:
    cistopic_obj = pickle.load(f)

print(cistopic_obj)
print()
print(cistopic_obj.cell_data["cell_type"].value_counts())
print()
for name, path in [("fragments", FRAGMENTS_PATH), ("chromsizes", CHROMSIZES)]:
    print(f"{'OK' if os.path.exists(path) else 'MISSING':8s} {name:12s} {path}")

CistopicObject from project 8k_mouse_cortex_ATACv2_nextgem_Chromium_Controller with n_cells × n_regions = 6111 × 1211806

Excitatory neurons    3436
Oligodendrocytes       711
Astrocytes             670
Inhibitory neurons     595
Microglia              350
OPC                    246
Endothelial            103
Name: cell_type, dtype: int64

OK       fragments    /home/jovyan/work/repo/notebooks/notebooks_PUMATAC/out/data/fragments/8k_mouse_cortex_ATACv2_nextgem_Chromium_Controller.fragments.tsv.gz
OK       chromsizes   /home/jovyan/work/PUMATAC_dependencies/genomes/mm10.chrom.sizes


In [3]:
# MACS2 receives the sample name through --name; names containing spaces
# break argument parsing, so cell type labels are converted to underscore
# form for the pseudobulk and peak calling steps.
cistopic_obj.cell_data["cell_type_macs"] = (
    cistopic_obj.cell_data["cell_type"].str.replace(" ", "_", regex=False)
)

print(cistopic_obj.cell_data["cell_type_macs"].value_counts())

Excitatory_neurons    3436
Oligodendrocytes       711
Astrocytes             670
Inhibitory_neurons     595
Microglia              350
OPC                    246
Endothelial            103
Name: cell_type_macs, dtype: int64


In [5]:
chromsizes = pd.read_csv(CHROMSIZES, sep="\t", header=None, names=["Chromosome", "End"])
chromsizes["Start"] = 0
chromsizes = chromsizes[["Chromosome", "Start", "End"]]

# export_pseudobulk returns (bigwig paths, bed paths), in that order
bw_paths, bed_paths = export_pseudobulk(
    input_data=cistopic_obj,
    variable="cell_type_macs",
    chromsizes=chromsizes,
    bed_path=BED_PATH,
    bigwig_path=BIGWIG_PATH,
    path_to_fragments={SAMPLE: FRAGMENTS_PATH},
    sample_id_col="sample_id",
    n_cpu=8,
    normalize_bigwig=True,
    split_pattern="___",
    temp_dir="/tmp",
)

with open(os.path.join(PSEUDOBULK_OUT, "bed_paths.pkl"), "wb") as f:
    pickle.dump(bed_paths, f, protocol=pickle.HIGHEST_PROTOCOL)
with open(os.path.join(PSEUDOBULK_OUT, "bw_paths.pkl"), "wb") as f:
    pickle.dump(bw_paths, f, protocol=pickle.HIGHEST_PROTOCOL)

print("BED files:")
for k, v in bed_paths.items():
    print(f"  {k:22s} {v}")
print("\nBigWig files:")
for k, v in bw_paths.items():
    print(f"  {k:22s} {v}")

2026-08-20 21:24:00,926 cisTopic     INFO     Splitting fragments by cell type.


[W::hts_idx_load3] The index file is older than the data file: /home/jovyan/work/repo/notebooks/notebooks_PUMATAC/out/data/fragments/8k_mouse_cortex_ATACv2_nextgem_Chromium_Controller.fragments.tsv.gz.tbi


2026-08-20 21:32:18,534 cisTopic     INFO     generating bigwig files


/home/jovyan/pycisTopic/src/pycisTopic/__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
/home/jovyan/pycisTopic/src/pycisTopic/__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
/home/jovyan/pycisTopic/src/pycisTopic/__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or 

BED files:
  Oligodendrocytes       /home/jovyan/work/repo/results/pseudobulk/bed/Oligodendrocytes.fragments.tsv.gz
  Excitatory_neurons     /home/jovyan/work/repo/results/pseudobulk/bed/Excitatory_neurons.fragments.tsv.gz
  Microglia              /home/jovyan/work/repo/results/pseudobulk/bed/Microglia.fragments.tsv.gz
  Inhibitory_neurons     /home/jovyan/work/repo/results/pseudobulk/bed/Inhibitory_neurons.fragments.tsv.gz
  OPC                    /home/jovyan/work/repo/results/pseudobulk/bed/OPC.fragments.tsv.gz
  Astrocytes             /home/jovyan/work/repo/results/pseudobulk/bed/Astrocytes.fragments.tsv.gz
  Endothelial            /home/jovyan/work/repo/results/pseudobulk/bed/Endothelial.fragments.tsv.gz

BigWig files:
  Oligodendrocytes       /home/jovyan/work/repo/results/pseudobulk/bigwig/Oligodendrocytes.bw
  Excitatory_neurons     /home/jovyan/work/repo/results/pseudobulk/bigwig/Excitatory_neurons.bw
  Microglia              /home/jovyan/work/repo/results/pseudobulk/bigwig/Mi

In [6]:
from pycisTopic.pseudobulk_peak_calling import peak_calling

narrow_peak_dict = peak_calling(
    macs_path="/opt/conda/bin/macs2",
    bed_paths=bed_paths,
    outdir=MACS_OUT,
    genome_size="mm",
    n_cpu=7,
    input_format="BEDPE",
    shift=73,
    ext_size=146,
    keep_dup="all",
    q_value=0.05,
)

with open(os.path.join(MACS_OUT, "narrow_peak_dict.pkl"), "wb") as f:
    pickle.dump(narrow_peak_dict, f, protocol=pickle.HIGHEST_PROTOCOL)

print("Peaks called per cell type:")
for cell_type in sorted(narrow_peak_dict.keys()):
    n_peaks = len(narrow_peak_dict[cell_type])
    print(f"  {cell_type:22s} {n_peaks:>8,} peaks")

2026-08-20 21:40:49,615	WARNING services.py:2248 -- WARNING: The object store is using /tmp/ray instead of /dev/shm because /dev/shm has only 67067904 bytes available. This will harm performance! You may be able to free up space by deleting files in /dev/shm. If you are inside a Docker container, you can increase /dev/shm size by passing '--shm-size=10.24gb' to 'docker run' (or add it to the run_options list in a Ray cluster config). Make sure to set this to more than 30% of available RAM.
2026-08-20 21:40:56,093	INFO worker.py:2024 -- Started a local Ray instance.
(pid=1297428) /home/jovyan/pycisTopic/src/pycisTopic/__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
(pid=1297428)   from pkg_resources import DistributionNotFound, get_distribution


(macs_call_peak_ray pid=1297428) 2026-08-20 21:41:05,178 cisTopic     INFO     Calling peaks for Inhibitory_neurons with /opt/conda/bin/macs2 callpeak --treatment /home/jovyan/work/repo/results/pseudobulk/bed/Inhibitory_neurons.fragments.tsv.gz --name Inhibitory_neurons  --outdir /home/jovyan/work/repo/results/macs2 --format BEDPE --gsize mm --qvalue 0.05 --nomodel --shift 73 --extsize 146 --keep-dup all --call-summits --nolambda
(macs_call_peak_ray pid=1297430) 2026-08-20 21:41:05,180 cisTopic     INFO     Calling peaks for Astrocytes with /opt/conda/bin/macs2 callpeak --treatment /home/jovyan/work/repo/results/pseudobulk/bed/Astrocytes.fragments.tsv.gz --name Astrocytes  --outdir /home/jovyan/work/repo/results/macs2 --format BEDPE --gsize mm --qvalue 0.05 --nomodel --shift 73 --extsize 146 --keep-dup all --call-summits --nolambda
(macs_call_peak_ray pid=1297429) 2026-08-20 21:41:05,422 cisTopic     INFO     Calling peaks for Microglia with /opt/conda/bin/macs2 callpeak --treatment /h

(pid=1297430) /home/jovyan/pycisTopic/src/pycisTopic/__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81. [repeated 6x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)
(pid=1297430)   from pkg_resources import DistributionNotFound, get_distribution [repeated 6x across cluster]


Peaks called per cell type:
  Astrocytes               98,091 peaks
  Endothelial              24,333 peaks
  Excitatory_neurons      295,912 peaks
  Inhibitory_neurons      143,827 peaks
  Microglia                89,642 peaks
  OPC                      69,212 peaks
  Oligodendrocytes         65,471 peaks
